# RQ3: do the metabolites behind a prediction survive compression?

A walkthrough of the attribution analysis in `analysis/`, run here on synthetic data by
importing and calling the real modules' own `main()` functions directly (not a
reimplementation): `explainability_stage1.py` (four attribution methods on the raw
metabolome), `explainability_stage2.py` (the same question after PCA/UMAP/VAE), and
`explainability_stage2_rf.py` (does the answer depend on using a random forest downstream
instead of logistic regression). All three write their usual CSV/Markdown outputs, here to
`results/explainability/`.

Fixed at 6 months, `fishoil` and `sex`, same as the study (see each script's own docstring).
Embedding dimensions are reduced to `{2, 16}` for speed (the study uses `{2, 3, 4, 8, 16}`).
The data in `data/` is purely synthetic, drawn from chosen statistical distributions, not
derived from any real data (see `00_build_data.ipynb`).


In [ ]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd().parent
SRC_DIR = REPO_ROOT / "src"
DATA_DIR = REPO_ROOT / "data"
sys.path.insert(0, str(SRC_DIR))
sys.path.insert(0, str(SRC_DIR / "analysis"))
# So a subprocess started from here (python "scripts/some_script.py") can "from metabo
# import ..." too, exactly as if the package had been pip installed.
os.environ["PYTHONPATH"] = str(SRC_DIR)
os.environ["METABO_DATA_DIR"] = str(DATA_DIR)
os.environ["METABO_RESULTS_DIR"] = str(Path.cwd() / "results")

# data/ is not shipped with the repo -- generate it on first use, same recipe as
# 00_build_data.ipynb, if it is not already there.
_RENAMED = {
    "datacorr_6m6y18mMw1w24_final.csv": "synthetic_raw_metabolome.csv",
    "target_for_metabolites_ml.csv": "synthetic_targets.csv",
    "metabolites_6m_for_ml.csv": "synthetic_metabolites_6m.csv",
    "metabolites_18m_for_ml.csv": "synthetic_metabolites_18m.csv",
    "metabolites_6y_for_ml.csv": "synthetic_metabolites_6y.csv",
}
if not all((DATA_DIR / new).exists() for new in _RENAMED.values()):
    import subprocess
    import shutil
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, str(SRC_DIR / "scripts" / "make_synthetic_data.py"),
                    "--outdir", str(DATA_DIR), "--force"], check=True, capture_output=True)
    subprocess.run([sys.executable, str(SRC_DIR / "scripts" / "build_inputs.py"),
                    "--source", str(DATA_DIR / "datacorr_6m6y18mMw1w24_final.csv"),
                    "--outdir", str(DATA_DIR)], check=True, capture_output=True)
    for old, new in _RENAMED.items():
        shutil.move(str(DATA_DIR / old), str(DATA_DIR / new))
    print("generated data/ (not shipped with the repo)")
# The synthetic data files are named distinctly from metabo.config's own default filenames,
# so every script that reads them -- in this process or in a subprocess started from here --
# is told explicitly which files to use.
os.environ["METABO_SOURCE"] = str(DATA_DIR / "synthetic_raw_metabolome.csv")
os.environ["METABO_TARGET_FILE"] = str(DATA_DIR / "synthetic_targets.csv")
os.environ["METABO_METAB_6M"] = str(DATA_DIR / "synthetic_metabolites_6m.csv")
os.environ["METABO_METAB_18M"] = str(DATA_DIR / "synthetic_metabolites_18m.csv")
os.environ["METABO_METAB_6Y"] = str(DATA_DIR / "synthetic_metabolites_6y.csv")
# The synthetic metabolites are named generically (metabolite001, ...), not with real
# chemical nomenclature, so the attribution scripts are told which generic name is the
# planted fish-oil marker (and its weaker parent compound).
os.environ["METABO_FISHOIL_MARKER"] = "metabolite001"
os.environ["METABO_FISHOIL_MARKER_PARENT"] = "metabolite002"
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("NUMBA_THREADING_LAYER", "workqueue")

# Reduced from the study's settings (1000 bootstraps/permutations, all five embedding
# dimensions, ten repeats) so every cell below finishes in well under a minute.
os.environ["METABO_N_BOOTSTRAP"] = "100"
os.environ["METABO_N_PERMUTATION"] = "200"
os.environ["METABO_EMBED_DIMS"] = "2 16"
os.environ["METABO_VAE_MAX_EPOCHS"] = "40"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from metabo import config, data_io, evaluation

print("embedding dims used here:", config.EMBED_DIMS, "(study default: 2 3 4 8 16)")


## Stage 1: attribution methods on the raw metabolome

Four methods (absolute elastic-net coefficient, permutation importance of the same logistic
model, permutation importance of a random forest, univariate mutual information), each
computed per outer fold and averaged, then ranked (rank 1 = most important). For `fishoil`,
the known planted marker (`metabolite001`) should rank at or near the top under every method.


In [ ]:
import explainability_stage1 as stage1

stage1.main()


In [ ]:
stage1_summary = pd.read_csv(config.RESULTS_DIR / "explainability" / "stage1_summary.csv")
stage1_summary[stage1_summary.outcome == "fishoil"][
    ["method", "cmpf_rank", "stability_spearman", "top20_jaccard"]
].round(3)


## Stage 2: does the marker survive compression?

The same logistic-regression coefficients, mapped back from each embedding to the metabolites
(PCA: the exact linear map; UMAP: a correlation-based surrogate; VAE: the decoder Jacobian, if
PyTorch is installed), compared with the Stage 1 raw ranking.


In [ ]:
import explainability_stage2 as stage2

stage2.main()


In [ ]:
stage2_df = pd.read_csv(config.RESULTS_DIR / "explainability" / "stage2_backprojection.csv")
cols = ["rep", "dim", "embedded_auroc", "spearman_vs_raw", "top20_jaccard_vs_raw",
        "hydroxyCMPF_backproj_rank", "hydroxyCMPF_raw_rank"]
stage2_df[stage2_df.outcome == "fishoil"][cols].round(3)


The planted marker's raw-metabolome rank (`hydroxyCMPF_raw_rank`, from Stage 1) should
be low (close to 1); `hydroxyCMPF_backproj_rank` after each embedding is usually much higher.

## Stage 2, with a different downstream model

Same question, but the embedded classifier is a random forest instead of logistic regression,
and the component-level importance is permutation importance (which has no sign, so the PCA
mapping here is the non-negative magnitude map, not the exact signed weights above).


In [ ]:
import explainability_stage2_rf as stage2_rf

stage2_rf.main()


In [ ]:
stage2_rf_df = pd.read_csv(config.RESULTS_DIR / "explainability" / "stage2_rf_backprojection.csv")
stage2_rf_df[stage2_rf_df.outcome == "fishoil"][
    ["rep", "dim", "embedded_auroc", "spearman_vs_raw", "hydroxyCMPF_backproj_rank"]
].round(3)


## Next steps

- `analysis/explainability_stage2_vae_encoder.py` attributes through the VAE's *encoder*
  instead of its decoder (a different, also-valid mapping) and can repeat the whole analysis
  over several fold seeds plus a shuffled-label null distribution with `--seeds`/`--null`
  (heavier; not run here).
- `analysis/explainability_stage2_fold_seeds.py` is the full repeated-seed, shuffled-label-null
  version behind the study's own robustness check (Table 6 / Figure 3); also heavier, not run
  here.


## Attribution through the VAE's encoder, and a robustness check over several fold seeds

Two further scripts round out the attribution analysis:

- `analysis/explainability_stage2_vae_encoder.py` attributes through the VAE's *encoder*
  (the gradient of the classifier's linear predictor with respect to the input metabolites,
  via automatic differentiation) rather than its decoder -- a different, also-valid mapping
  from the same fitted model.
- `analysis/explainability_stage2_fold_seeds.py` (PCA/UMAP) and the `--seeds`/`--null` modes
  of the encoder-gradient script above repeat the attribution over several outer-fold seeds,
  and over several runs with the outcome label shuffled, to check that the fish-oil marker's
  recovery is not a one-seed accident and is better than chance.

Both scripts are run here as real subprocesses (not reimplemented), with far fewer seeds and
null runs than the study (2 seeds, 5 null runs here; the study uses 10 seeds and 100 null
runs) purely for speed. Skipped entirely if PyTorch is not installed.


In [ ]:
import re
import subprocess
import sys

try:
    import torch  # noqa: F401
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
    print("torch not installed -- skipping this section (pip install torch to enable)")


def hide_local_paths(text):
    text = text.replace(str(REPO_ROOT), ".")
    return re.sub(r"(/home|/Users)/[^/]+", "~", text)


def run_script(name, *args):
    cmd = [sys.executable, str(SRC_DIR / "analysis" / name), *args]
    r = subprocess.run(cmd, cwd=SRC_DIR, capture_output=True, text=True)
    print(f"$ analysis/{name} {' '.join(args)}")
    print(hide_local_paths(r.stdout)[-1500:])
    if r.returncode != 0:
        print(hide_local_paths(r.stderr)[-2000:])
    r.check_returncode()


In [ ]:
if HAS_TORCH:
    run_script("explainability_stage2_vae_encoder.py")
    run_script("explainability_stage2_vae_encoder.py", "--seeds", "42", "43", "--vary", "both")
    run_script("explainability_stage2_vae_encoder.py", "--null", "5")


In [ ]:
run_script("explainability_stage2_fold_seeds.py", "--seeds", "42", "43")
run_script("explainability_stage2_fold_seeds.py", "--null", "5")


Both robustness scripts write directly to `OUTDIR` (this notebook's `results/explainability/`); their own summarizers read those files back.

In [ ]:
if HAS_TORCH:
    run_script("summarize_encoder_seeds.py")


In [ ]:
if HAS_TORCH:
    run_script("summarize_fold_seeds.py")


In [ ]:
if HAS_TORCH:
    fold_summary = pd.read_csv(config.RESULTS_DIR / "explainability" / "stage2_fold_seeds_summary.csv")
    fold_summary[fold_summary.outcome == "fishoil"][
        ["rep", "dim", "n_runs", "rank_median", "rank_min", "rank_max", "spearman_median"]
    ].round(3)


Fish oil's planted-marker rank should land in roughly the same range across the 2 fold
seeds tried here as it did for the single seed used earlier in this notebook -- i.e. the
degradation under compression is not an artifact of one particular cross-validation split.

## Next steps

- Rerun both robustness scripts with the study's own seed count (`--seeds 42 43 44 45 46 47
  48 49 50 51` -- i.e. 10 seeds) and null-run count (`--null 100`) for a precise estimate;
  expect this to take much longer than the reduced version above.
